# Lab 6: Giới thiệu về Transformers

Họ và tên: Đỗ Thảo Giang

Msv: 22001253

In [ ]:
!pip install transformers torch

## Bài 1: Khôi phục Masked Token (Masked Language Modeling)

In [ ]:
from transformers import pipeline
# 1. Tải pipeline "fill-mask"
# Pipeline này sẽ tự động tải một mô hình mặc định phù hợp (thường là một biến thể của BERT)
mask_filler = pipeline("fill-mask")
# 2. Câu đầu vào với token [MASK]
input_sentence = "Hanoi is the <mask> of Vietnam."
# 3. Thực hiện dự đoán
# top_k=5 yêu cầu mô hình trả về 5 dự đoán hàng đầu
predictions = mask_filler(input_sentence, top_k=5)
# 4. In kết quả
print(f"Câu gốc: {input_sentence}")
for pred in predictions:
  print(f"Dự đoán: '{pred['token_str']}' với độ tin cậy: {pred['score']:.4f}")
  print(f" -> Câu hoàn chỉnh: {pred['sequence']}")

No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8 (https://huggingface.co/distilbert/distilroberta-base).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Some weights of the model checkpoint at distilbert/distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


Câu gốc: Hanoi is the <mask> of Vietnam.
Dự đoán: ' capital' với độ tin cậy: 0.9341
 -> Câu hoàn chỉnh: Hanoi is the capital of Vietnam.
Dự đoán: ' Republic' với độ tin cậy: 0.0300
 -> Câu hoàn chỉnh: Hanoi is the Republic of Vietnam.
Dự đoán: ' Capital' với độ tin cậy: 0.0105
 -> Câu hoàn chỉnh: Hanoi is the Capital of Vietnam.
Dự đoán: ' birthplace' với độ tin cậy: 0.0054
 -> Câu hoàn chỉnh: Hanoi is the birthplace of Vietnam.
Dự đoán: ' heart' với độ tin cậy: 0.0014
 -> Câu hoàn chỉnh: Hanoi is the heart of Vietnam.


1. Mô hình đã dự đoán chính xác từ "capital" với độ chính xác 0.9341.

2. Mô hình Encoder-only phù hợp nhất vì khả năng "nhìn thấy" toàn bộ ngữ cảnh hai chiều (Bidirectional Context).

(a) Encoder-only được thiết kế để hiểu ngữ cảnh hai phía (bidirectional context)

- BERT và các mô hình encoder-only được huấn luyện theo cơ chế:

    + Masked Language Modeling (MLM): che ngẫu nhiên 15% token và yêu cầu mô hình dự đoán chúng.

- Mỗi token được mã hóa dựa trên cả trái và phải nên rất phù hợp để dự đoán từ bỏ trống trong câu.

(b) Huấn luyện ban đầu (pre-training) của BERT đã dùng chính tác vụ mask

-  Fill-mask là tác vụ mô hình được huấn luyện gốc, nên hiệu quả rất cao.

(c) Encoder chỉ tập trung vào biểu diễn (representation learning)

- BERT không tự sinh văn bản, mà chỉ hiểu và mã hóa câu.

(d) Mô hình decoder (GPT) không thể làm fill-mask tự nhiên

- GPT không hỗ trợ token [MASK], vì nó chỉ nhìn được ngữ cảnh từ trái → phải.

- Nên GPT không thể xem đồng thời hai phía của khoảng trống để điền vào.

## Bài 2: Dự đoán từ tiếp theo (Next Token Prediction)

In [ ]:
from transformers import pipeline
# 1. Tải pipeline "text-generation"
# Pipeline này sẽ tự động tải một mô hình phù hợp (thường là GPT-2)
generator = pipeline("text-generation")
# 2. Đoạn văn bản mồi
prompt = "The best thing about learning NLP is"
# 3. Sinh văn bản
# max_length: tổng độ dài của câu mồi và phần được sinh ra
# num_return_sequences: số lượng chuỗi kết quả muốn nhận
generated_texts = generator(prompt, max_length=50, num_return_sequences=1)
# 4. In kết quả
print(f"Câu mồi: '{prompt}'")
for text in generated_texts:
  print("Văn bản được sinh ra:")
  print(text['generated_text'])

No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d (https://huggingface.co/openai-community/gpt2).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Câu mồi: 'The best thing about learning NLP is'
Văn bản được sinh ra:
The best thing about learning NLP is that the training is easy. You just press a button for a few seconds and get an up-close and personal view of all the different things that you do. It's like a real NLP class, but with a completely different approach.

Your job as a teacher should be to teach what you love. Your job as a teacher is to teach others what you love.

If you want to get your teacher's attention, you should do it from the start. You should make sure that they understand what you're saying and your message.

If you want to build a solid relationship with your teacher, you should talk to her about things that she's taught you.

They should be able to listen to you and understand what you're trying to say. They should know what you're trying to say.

They should know what you're trying to say. If they're not able to listen to you or not able to listen to you, you might feel pressured.

If you want to build

1. Kết quả có hợp lý– ở mức ổn đối với GPT-2, nhưng không thật sự xuất sắc.

Giải thích chi tiết:

Mô hình GPT-2, một mô hình cũ (ra mắt 2019) và nhỏ so với các mô hình hiện nay → chất lượng sinh văn bản không thể so sánh với GPT-3.5, GPT-4, hay LLaMA-3.

Văn bản sinh ra:

Có mạch ý, ít nhất là liên quan đến learning NLP trong vài câu đầu.

Nhưng bắt đầu lặp ý, lan man, và mất mạch văn, ví dụ:

"They are friends. They are parents. They are friends."

Lỗi dễ lặp lại chuỗi khi không điều chỉnh tham số sinh (temperature, repetition_penalty).

Kết luận ngắn: Hợp lý ở mức mô hình nhỏ.

2. Tại sao các mô hình Decoder-only như GPT phù hợp cho tác vụ Text Generation?

Mô hình Decoder-only chỉ có bộ giải mã được thiết kế đúng theo cơ chế sinh văn bản từng từ vì vậy rất tối ưu cho các tác vụ sinh văn bản.

- Kiến trúc tự hồi quy (autoregressive) Decoder-only hoạt động theo cơ chế: Sinh token hiện tại dựa trên tất cả token trước đó.

- Không cần Encoder để hiểu câu. Khác với kiến trúc Encoder–Decoder (như BART, T5):

    + Encoder xử lý input

    + Decoder sinh output dựa trên embedding từ encoder

- Tối ưu cho tốc độ suy luận: Decoder-only chỉ xử lý một chiều (từ trái sang phải) nên:
  + Nhanh hơn encoder-decoder
  + Tiêu tốn ít tài nguyên hơn
  + Phù hợp cho sinh văn dài

- Được pretrain trên dạng dự đoán token tiếp theo


- Dễ điều chỉnh với tham số sinh (sampling). Các mô hình GPT hỗ trợ nhiều kỹ thuật:

  + temperature

  + top-k sampling

  + top-p sampling

  + repetition penalty

Từ đó giúp kiểm soát phong cách sinh văn bản (sáng tạo, lặp ít hơn…).

## Bài 3: Tính toán Vector biểu diễn của câu (Sentence Representation)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

# 1. Chọn một mô hình BERT
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
# 2. Câu đầu vào
sentences = ["This is a sample sentence."]
# 3. Tokenize câu
# padding=True: đệm các câu ngắn hơn để có cùng độ dài
# truncation=True: cắt các câu dài hơn
# return_tensors='pt': trả về kết quả dưới dạng PyTorch tensors
inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')
# 4. Đưa qua mô hình để lấy hidden states
# torch.no_grad() để không tính toán gradient, tiết kiệm bộ nhớ
with torch.no_grad():
  outputs = model(**inputs)
# outputs.last_hidden_state chứa vector đầu ra của tất cả các token
last_hidden_state = outputs.last_hidden_state
# shape: (batch_size, sequence_length, hidden_size)
# 5. Thực hiện Mean Pooling
# Để tính trung bình chính xác, chúng ta cần bỏ qua các token đệm (padding tokens)
attention_mask = inputs['attention_mask']
mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
sum_embeddings = torch.sum(last_hidden_state * mask_expanded, 1)
sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
sentence_embedding = sum_embeddings / sum_mask
# 6. In kết quả
print("Vector biểu diễn của câu:")
print(sentence_embedding)
print("\nKích thước của vector:", sentence_embedding.shape)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Vector biểu diễn của câu:
tensor([[-6.3874e-02, -4.2837e-01, -6.6779e-02, -3.8430e-01, -6.5784e-02,
         -2.1826e-01,  4.7636e-01,  4.8659e-01,  4.0647e-05, -7.4273e-02,
         -7.4740e-02, -4.7635e-01, -1.9773e-01,  2.4824e-01, -1.2162e-01,
          1.6678e-01,  2.1045e-01, -1.4576e-01,  1.2636e-01,  1.8635e-02,
          2.4640e-01,  5.7090e-01, -4.7014e-01,  1.3782e-01,  7.3650e-01,
         -3.3808e-01, -5.0331e-02, -1.6452e-01, -4.3517e-01, -1.2900e-01,
          1.6516e-01,  3.4004e-01, -1.4930e-01,  2.2422e-02, -1.0488e-01,
         -5.1916e-01,  3.2964e-01, -2.2162e-01, -3.4206e-01,  1.1993e-01,
         -7.0148e-01, -2.3126e-01,  1.1224e-01,  1.2550e-01, -2.5191e-01,
         -4.6374e-01, -2.7261e-02, -2.8415e-01, -9.9249e-02, -3.7017e-02,
         -8.9192e-01,  2.5005e-01,  1.5816e-01,  2.2701e-01, -2.8497e-01,
          4.5300e-01,  5.0945e-03, -7.9441e-01, -3.1008e-01, -1.7403e-01,
          4.3029e-01,  1.6816e-01,  1.0590e-01, -4.8987e-01,  3.1856e-01,
          3.

1. Kích thước chiều của Vector Biểu diễn

Kích thước (chiều) của vector biểu diễn là 768.
- Tương ứng với tham số: Con số này tương ứng với tham số $hidden\_size$ (hoặc $d_{\text{model}}$) của mô hình BERT, cụ thể là bert-base-uncased.

- Ý nghĩa: $hidden\_size$ là số chiều của vector đầu ra cho mỗi token (còn gọi là vector nhúng hoặc token embedding). Vector biểu diễn câu cuối cùng được tính bằng cách lấy trung bình các vector đầu ra này (trừ các token đệm), nên nó vẫn giữ nguyên số chiều là 768.

2. Lý do cần sử dụng attention_mask khi thực hiện Mean Pooling

Chúng ta cần sử dụng attention_mask (mặt nạ chú ý) khi thực hiện Mean Pooling để đảm bảo các token đệm (padding tokens) không được tính vào phép tính trung bình của câu.

- Mô hình BERT cần đệm (pad) các câu ngắn hơn bằng token [PAD] để tất cả các câu trong một batch có cùng độ dài.

- attention_mask đánh dấu 1 cho các token thực và 0 cho các token đệm.

- Khi nhân vector đầu ra của mô hình với mask, các vector nhúng của token đệm trở thành zero và không còn ảnh hưởng đến giá trị cuối cùng khi bạn tính tổng và chia trung bình.